In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import shap
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, precision_recall_curve,
)

RANDOM_STATE = 42
CHEMIN_DONNEES = "/home/moctar04/Documents/data/data_flow_project/Projet-Dataflow360/notebooks/dataset_fraude_clean.csv"  # <-

In [5]:

# ---------------------------------------------------------------------------
# 1. Chargement et préparation
# ---------------------------------------------------------------------------
df = pd.read_csv(CHEMIN_DONNEES)
 
colonnes_bool = ["nouvel_appareil", "nouveau_destinataire"]
for c in colonnes_bool:
    df[c] = df[c].astype(int)
 
CIBLE = "fraude"
FEATURES_NUM = [
    "montant", "heure", "ecart_montant_moyen", "ecart_heure_habituelle",
] + colonnes_bool
FEATURES_CAT = ["type", "canal"]
FEATURES = FEATURES_NUM + FEATURES_CAT
 
X = df[FEATURES]
y = df[CIBLE].astype(int)
 
print(f"Lignes : {len(df):,} | Taux de fraude global : {y.mean():.3%}")
 

Lignes : 493,604 | Taux de fraude global : 2.506%


nous notons un taux tres faible de fraude présent dans le dataset ce qui va necessité un "rééquilibrage"

In [ ]:
# ---------------------------------------------------------------------------
# 2. Séparation train/test
# ---------------------------------------------------------------------------
# IMPORTANT : ceci est un split ALÉATOIRE STRATIFIÉ, pas temporel, faute de
# colonne de date/timestamp explicite dans cet extrait. Comme on l'a vu dans
# la méthodologie, un split temporel (entraîner sur le passé, tester sur le
# plus récent) est préférable pour éviter toute fuite implicite et obtenir
# une estimation fidèle de la performance en production. Si le dataset
# complet contient une vraie colonne de date/heure d'événement (pas juste
# "heure" qui ne représente que l'heure de la journée), remplacer ce split
# par un tri chronologique + découpage séquentiel.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train : {len(X_train):,} ({y_train.mean():.3%} fraude) | "
      f"Test : {len(X_test):,} ({y_test.mean():.3%} fraude)")

Train : 394,883 (2.506% fraude) | Test : 98,721 (2.506% fraude)


stratify=y nous permet de conserver approximativement la même proportion de fraudes dans train et test.

In [8]:
# ---------------------------------------------------------------------------
# 3. Préprocesseur commun (encodage des variables catégorielles)
# ---------------------------------------------------------------------------
preprocesseur = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT)],
    remainder="passthrough",
)
 
def noms_features_apres_encodage(preprocesseur_fit) -> list[str]:
    """Récupère les noms de colonnes après OneHotEncoding, pour SHAP et le feature importance."""
    noms_cat = list(preprocesseur_fit.named_transformers_["cat"].get_feature_names_out(FEATURES_CAT))
    return noms_cat + FEATURES_NUM
 

In [9]:

# ---------------------------------------------------------------------------
# 4. Entraînement + tracking MLflow pour chaque modèle
# ---------------------------------------------------------------------------
ratio_desequilibre = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
 
MODELES = {
    "logistic_regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300, max_depth=None, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "xgboost": XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        scale_pos_weight=ratio_desequilibre, eval_metric="aucpr",
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
}
 
mlflow.set_experiment("dataflow360_detection_fraude")
 
 
def entrainer_et_logger(nom_modele: str, modele) -> tuple[Pipeline, dict]:
    with mlflow.start_run(run_name=nom_modele):
        pipe = Pipeline([("prep", preprocesseur), ("model", modele)])
        pipe.fit(X_train, y_train)
 
        proba_test = pipe.predict_proba(X_test)[:, 1]
        preds_test = (proba_test >= 0.5).astype(int)
 
        metriques = {
            "precision": precision_score(y_test, preds_test, zero_division=0),
            "recall": recall_score(y_test, preds_test, zero_division=0),
            "f1": f1_score(y_test, preds_test, zero_division=0),
            "roc_auc": roc_auc_score(y_test, proba_test),
            "pr_auc": average_precision_score(y_test, proba_test),  # métrique clé en cas de fort déséquilibre
        }
 
        mlflow.log_param("modele", nom_modele)
        mlflow.log_params({f"hp_{k}": v for k, v in modele.get_params().items()})
        mlflow.log_metrics(metriques)
        mlflow.sklearn.log_model(pipe, artifact_path="model", serialization_format="pickle")
 
        tn, fp, fn, tp = confusion_matrix(y_test, preds_test).ravel()
        print(f"\n=== {nom_modele} ===")
        for k, v in metriques.items():
            print(f"  {k:10s} : {v:.4f}")
        print(f"  Matrice de confusion -> VN={tn} FP={fp} FN={fn} VP={tp}")
 
        return pipe, metriques
 
 
resultats = {}
for nom, modele in MODELES.items():
    pipe, metriques = entrainer_et_logger(nom, modele)
    resultats[nom] = {"pipeline": pipe, "metriques": metriques}
 

2026/09/30 12:37:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/30 12:37:50 INFO mlflow.store.db.utils: Updating database tables
2026/09/30 12:37:52 INFO mlflow.tracking.fluent: Experiment with name 'dataflow360_detection_fraude' does not exist. Creating a new experiment.
/home/moctar04/Documents/data/data_flow_project/Projet-Dataflow360/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/09/30 12:39:16 WARNING mlflow.models.m


=== logistic_regression ===
  precision  : 0.5232
  recall     : 0.9511
  f1         : 0.6751
  roc_auc    : 0.9933
  pr_auc     : 0.9200
  Matrice de confusion -> VN=94103 FP=2144 FN=121 VP=2353


2026/09/30 12:40:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/30 12:40:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== random_forest ===
  precision  : 0.8565
  recall     : 0.8783
  f1         : 0.8673
  roc_auc    : 0.9899
  pr_auc     : 0.9307
  Matrice de confusion -> VN=95883 FP=364 FN=301 VP=2173


2026/09/30 12:40:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/30 12:40:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== xgboost ===
  precision  : 0.5331
  recall     : 0.9568
  f1         : 0.6847
  roc_auc    : 0.9950
  pr_auc     : 0.9420
  Matrice de confusion -> VN=94174 FP=2073 FN=107 VP=2367
